# Acquiring Lyrics for Songs

In [21]:
import pandas as pd
import requests
import json
import time

In [2]:
spotify_df = pd.read_csv("spotify_songs.csv")

In [17]:
reduced_spotify_df = spotify_df.drop_duplicates(subset=["track_name", "track_album_name"])

In [9]:
sample_track_name = spotify_df["track_name"].iloc[0]
sample_track_album_name = spotify_df["track_album_name"].iloc[0]
sample_track_artist = spotify_df["track_artist"].iloc[0]
sample_track_duration = round(spotify_df["duration_ms"].iloc[0] / 1000)

In [10]:
sample_track_duration

195

In [ ]:
url = "https://lrclib.net/api/get"

In [16]:
params = {
    "artist_name": sample_track_artist,
    "track_name": sample_track_name,
    "album_name": sample_track_album_name,
    "duration": sample_track_duration
}

response = requests.get(
    "https://lrclib.net/api/get",
    params=params,
    headers={"User-Agent": "SpotifyAnalysis/1.0"},
    timeout=15
)

if response.status_code == 200:
    data = response.json()
    lyrics = data.get("plainLyrics")
    print(lyrics)
else:
    print(response.status_code, response.text)

I'm at a party I don't wanna be at
And I don't ever wear a suit and tie, yeah
Wondering if I could sneak out the back
Nobody's even looking me in my eyes
Then you take my hand
Finish my drink, say, "Shall we dance?" (Hell, yeah)
You know I love ya, did I ever tell ya?
You make it better like that

Don't think I fit in at this party
Everyone's got so much to say, yeah
I always feel like I'm nobody, mmm
Who wants to fit in anyway?

'Cause I don't care when I'm with my baby, yeah
All the bad things disappear
And you're making me feel like maybe I am somebody
I can deal with the bad nights
When I'm with my baby, yeah
(Ooh, ooh, ooh, ooh, ooh, ooh)

'Cause I don't care as long as you just hold me near
You can take me anywhere
And you're making me feel like I'm loved by somebody
I can deal with the bad nights
When I'm with my baby, yeah
(Ooh, ooh, ooh, ooh, ooh, ooh)

(Ooh, ooh, ooh, ooh, ooh, ooh)
'Cause I don't care

(Ooh, ooh, ooh, ooh, ooh, ooh)
'Cause I don't care

We at a party we don'

In [22]:
with open("lyrics.jsonl", "a", encoding="utf-8") as f_success, \
     open("failed_lyrics.jsonl", "a", encoding="utf-8") as f_failed:

    for _, row in reduced_spotify_df.iterrows():

        params = {
            "artist_name": row["track_artist"],
            "track_name": row["track_name"],
            "album_name": row["track_album_name"],
            "duration": round(row["duration_ms"] / 1000)
        }

        try:
            response = requests.get(
                "https://lrclib.net/api/get",
                params=params,
                headers={"User-Agent": "SpotifyAnalysis/1.0"},
                timeout=15
            )

            if response.status_code == 200:
                data = response.json()

                song_record = {
                    "track_id": row["track_id"],
                    "track_name": row["track_name"],
                    "artist_name": row["track_artist"],
                    "plainLyrics": data.get("plainLyrics"),
                    "syncedLyrics": data.get("syncedLyrics")
                }

                f_success.write(
                    json.dumps(song_record, ensure_ascii=False) + "\n"
                )

            else:
                failed_record = {
                    "track_id": row["track_id"],
                    "track_name": row["track_name"],
                    "artist_name": row["track_artist"],
                    "status_code": response.status_code,
                    "reason": response.reason
                }

                f_failed.write(
                    json.dumps(failed_record, ensure_ascii=False) + "\n"
                )

        except requests.RequestException as e:

            failed_record = {
                "track_id": row["track_id"],
                "track_name": row["track_name"],
                "artist_name": row["track_artist"],
                "status_code": None,
                "reason": str(e)
            }

            f_failed.write(
                json.dumps(failed_record, ensure_ascii=False) + "\n"
            )

        time.sleep(1.5)